In [17]:
!pip install wandb tqdm -q

In [18]:
import math
import time
import wandb
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.nn import functional as F

In [19]:
# =========================
# TRAINING CONFIG
# =========================

batch_size = 64
block_size = 256

max_iters = 15000
eval_interval = 500
eval_iters = 200

learning_rate = 2e-4
weight_decay = 0.1

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# MODEL
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.3

# GENERATION
temperature = 0.9
top_k = 40

# TRAINING
grad_clip = 1.0

torch.manual_seed(1337)

print(device)

cuda


In [20]:
wandb.login(key="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")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [21]:
wandb.init(
    project="mini-gpt",
    config={
        "batch_size": batch_size,
        "block_size": block_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
        "learning_rate": learning_rate,
    }
)

iter,▁▁▂▃▃▄▅▆▆▇█
lr,████▇▇▆▅▄▂▁
train_loss,██▃▃▂▂▂▂▂▁▁
val_loss,██▂▁▁▁▁▁▁▂▂
iter,4500
lr,0.00024
train_loss,0.45233
val_loss,1.9582


In [22]:
with open('/kaggle/input/datasets/bhautik04/input-data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Dataset length:", len(text))
print(text[:500])

Dataset length: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [23]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print(vocab_size)

65


In [24]:
def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x.to(device), y.to(device)

In [25]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()

    return out

In [26]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        # FIXED scaling
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)

        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # weight tying
        self.lm_head.weight = self.token_embedding_table.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=device)
        )

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None

        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, label_smoothing=0.1)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            logits = logits / temperature

            # top-k sampling
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [27]:
model = GPT().to(device)

# HUGE SPEEDUP on Kaggle
# model = torch.compile(model)

print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")

10.763969 M parameters


In [28]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
    betas=(0.9, 0.95)
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max_iters
)

scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_57/1472058727.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [29]:
best_val_loss = float('inf')
pbar = tqdm(
    range(max_iters),
    ascii=True,
    dynamic_ncols=True
)

for iter in pbar:
    if iter % eval_interval == 0:
        losses = estimate_loss()
        train_loss = losses['train']
        val_loss = losses['val']

        print(
            f"step {iter} | "
            f"train loss {train_loss:.4f} | "
            f"val loss {val_loss:.4f}"
        )

        wandb.log({
            "iter": iter,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": scheduler.get_last_lr()[0]
        })

        # save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                model.state_dict(),
                "best_model.pt"
            )

    xb, yb = get_batch('train')
    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(
        device_type='cuda',
        dtype=torch.float16
    ):

        logits, loss = model(xb, yb)
    
    scaler.scale(loss).backward()

    # gradient clipping
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        grad_clip
    )

    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
    pbar.set_postfix(loss=loss.item())

  0%|          | 0/15000 [00:00<?, ?it/s]

step 0 | train loss 4.2044 | val loss 4.2076
step 500 | train loss 2.5209 | val loss 2.5793
step 1000 | train loss 2.1557 | val loss 2.2931
step 1500 | train loss 2.0050 | val loss 2.1636
step 2000 | train loss 1.9230 | val loss 2.0968
step 2500 | train loss 1.8636 | val loss 2.0581
step 3000 | train loss 1.8288 | val loss 2.0298
step 3500 | train loss 1.7989 | val loss 2.0161
step 4000 | train loss 1.7604 | val loss 1.9921
step 4500 | train loss 1.7370 | val loss 1.9877
step 5000 | train loss 1.7145 | val loss 1.9824
step 5500 | train loss 1.6948 | val loss 1.9856
step 6000 | train loss 1.6735 | val loss 1.9812
step 6500 | train loss 1.6566 | val loss 1.9836
step 7000 | train loss 1.6345 | val loss 1.9814
step 7500 | train loss 1.6177 | val loss 1.9835
step 8000 | train loss 1.6049 | val loss 1.9915
step 8500 | train loss 1.5869 | val loss 1.9868
step 9000 | train loss 1.5760 | val loss 1.9989
step 9500 | train loss 1.5635 | val loss 1.9995
step 10000 | train loss 1.5493 | val loss 2.

In [30]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated = model.generate(
    context,
    max_new_tokens=1000
)[0].tolist()

print(decode(generated))


do not stand of the young prXerence with Angelo
For pretty captive and hand the councils
Of a brain temper; from the wars have proclaim'd
To her mortal ten things from the king:
And lose her brother brother York in Surrey,
Her father Earl of Buckingham, you, Warwick.

CLARENCE:
Alas, they
Do are notv
To bring the friends of such things in my soul.

WARWICK:
But I here uncle, my lordVin; and they love me.

YORK:
I entreat your grace to to prove it the truth.

KING EZETH:
Now, afore I have protest, that e'er now might
Ne'er in the roIt speak of sympathing boy;
And, too wise, and by my sight and good degree
Before joy the potion of peace of the Tower.
Come, let us suspect what a cause have done,
I will leave thee while: and thou shalt mark at thy hand3
LIET:
Hear me, my lord, then: so we no& KING told thee where thou prophetvest.

NORTHUMBERLAND:
Yes, if I did.

HENRY BOLINGBROKE:
Nore, maFether more inews, and return his embrace.

KING RICHARD II:
WhJack, what stay'st thou? foBlench of 

In [31]:
wandb.finish()

iter,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train_loss,█▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
iter,14500
lr,0.0
train_loss,1.50357
val_loss,2.01108


In [32]:
from IPython.display import FileLink

FileLink("best_model.pt")

/kaggle/working/best_model.pt